# Stellar Classification: Machine Learning Tutorial

## Overview

This tutorial demonstrates machine learning classification techniques using real astronomical data from the **Sloan Digital Sky Survey (SDSS)**. The goal is to classify stars as either:
- **Class 0**: Normal Stars
- **Class 1**: RR Lyrae Variable Stars

### What are RR Lyrae Stars?

**RR Lyrae stars** are an important class of variable stars that exhibit regular, short-period brightness oscillations (typically 0.2-1 day periods). They are:
- **Evolutionary markers**: Found primarily in old stellar populations
- **Distance indicators**: Used to measure distances to nearby galaxies and globular clusters
- **Scientific importance**: Help us understand stellar evolution and galaxy structure

### Dataset Description

We use the **RR Lyrae Combined Dataset** from the AstroML library, which contains photometric observations from the SDSS. The dataset includes:

**Features** (4 photometric color indices):
- **u-g**
- **g-r**
- **r-i**
- **i-z**

These color differences encode information about stellar temperature, composition, and physical properties, allowing us to distinguish RR Lyrae stars from normal stars.

**Class Distribution**: Highly imbalanced (~96% normal stars, ~4% RR Lyrae) - realistic for astronomical surveys

### Tutorial Structure

This tutorial progresses through several classification algorithms:

1. **kNN (k-Nearest Neighbors)** - Simple baseline, explores effect of k parameter
2. **Decision Trees** - Interpretable rules, explores tree depth effects
3. **Random Forests** - Ensemble method with feature importance analysis
4. **Grid Search Optimization** - Systematic hyperparameter tuning using cross-validation

### Key Concepts Covered

- **Hyperparameter tuning**: How to optimize model parameters for better performance
- **Cross-validation**: k-fold CV for robust evaluation on limited data
- **Imbalanced classification**: Handling datasets with unequal class distributions
- **Model evaluation metrics**: Accuracy, balanced accuracy, precision, recall, F1-score
- **Overfitting detection**: Comparing training vs. testing performance
- **Feature importance**: Understanding which features matter most for classification

### References

AstroML Library Documentation: https://www.astroml.org/modules/generated/astroML.datasets.fetch_rrlyrae_combined.html

---

## Code Documentation & Workflow

The notebook follows this workflow:

1. **Import Libraries** - Load required packages for data processing and machine learning
2. **Load Dataset** - Fetch the RR Lyrae combined dataset from AstroML
3. **Data Exploration** - Convert to pandas DataFrame, visualize distributions
4. **Data Preparation** - Create train/test splits with stratification to maintain class ratios
5. **Algorithm Testing** - Train multiple classifiers with hyperparameter variations
        
    5.1 **Evaluation** - Generate confusion matrices, classification reports, and visualizations

    5.2 **Optimization** - Use grid search for systematic hyperparameter tuning

6. **Comparison** - Compare algorithm performance across multiple metrics

--- 
# Import

In [ ]:
from astroML.datasets import fetch_rrlyrae_combined
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, 
                             confusion_matrix, classification_report, 
                             recall_score, precision_score, f1_score)
import numpy as np
from sklearn.model_selection import GridSearchCV

# Read in the dataset

In [ ]:
# Read combined RR Lyrae stars from the SDSS S82 catalog
rrlyrae_combined_data = fetch_rrlyrae_combined()

In [ ]:
# Extract features from the combined RR Lyrae dataset
X = rrlyrae_combined_data[0]
# Extract labels from the combined RR Lyrae dataset
y = rrlyrae_combined_data[1]

In [ ]:
X.shape, y.shape

# Explore the dataset

For this task, we will transform the dataset to a pandas DataFrame. 

In [ ]:
# Create a dataframe with features X and labels y
df = pd.DataFrame(X, columns=['u-g', 'g-r', 'r-i', 'i-z'])
df['Star_type'] = y

In [ ]:
# Drop part of the normal stars since they are overrepresented in the dataset
normal_stars = df[df['Star_type'] == 0]
normal_stars = normal_stars.sample(frac=0.4, random_state=42)  # Keep only 40% of normal stars
df = pd.concat([df[df['Star_type'] != 0], normal_stars])  # Combine with the rest of the dataset

In [ ]:
# Create a corner plot with scatter plots colored by star type

----

# kNN

## Introduction to k-Nearest Neighbors (kNN)

The **k-Nearest Neighbors** algorithm is one of the simplest yet powerful classification algorithms in machine learning. Here's how it works:

1. **Core Idea**: To classify a new data point, kNN finds the k closest training examples and assigns the label that appears most frequently among those neighbors.

2. **Distance Metric**: By default, it uses Euclidean distance, which measures the straight-line distance between points in feature space.

3. **Why Test Different k Values?**
   - **Small k (e.g., k=1)**: The model is very flexible and sensitive to individual data points. It may overfit the training data.
   - **Large k (e.g., k=30)**: The model becomes more rigid and may underfit, missing important patterns.
   - **Optimal k**: Balances between fitting the training data well and generalizing to new unseen data.

In this tutorial, we'll explore how different k values affect the classification of stars:
- **Class 0**: Normal stars
- **Class 1**: RR Lyrae variable stars (which have distinctive brightness variations)

These star types have different color patterns in the four photometric bands (u, g, r, i, z), which we'll use as features.

### Step 1: Prepare the Data

We split our dataset into two parts:
- **Training Set (80%)**: Used to train the kNN model by building the index of training samples
- **Testing Set (20%)**: Used to evaluate how well the model generalizes to unseen data

The class distribution tells us if the dataset is balanced or if one class dominates.

In [ ]:
# Split the data into training and testing sets
# IMPORTANT: Use stratified split to maintain class distribution
X = df[['u-g', 'g-r', 'r-i', 'i-z']]
y = df['Star_type']

### Step 2: Hyperparameter Tuning - Testing Different k Values

Here we systematically test k values from 1 to 40 and measure the accuracy on both training and testing sets.

- **Training Accuracy**: How well the model fits the training data
- **Testing Accuracy**: How well the model generalizes to new, unseen data (more important!)

A large gap between training and testing accuracy indicates **overfitting** - the model memorized the training data but doesn't generalize well.

In [ ]:
# Test different values of k
k_values = range(1, 41)
train_accuracies = []
test_accuracies = []

### Step 3: Visualize the Effect of k on Model Performance

This plot shows:
- **Blue line with circles**: Training accuracy across different k values (usually decreases as k increases)
- **Orange line with squares**: Testing accuracy (what we care about most)

Look for:
- The k value where testing accuracy is highest (the "elbow" or peak)
- The gap between training and testing accuracy (smaller gap = better generalization)
- The point where testing accuracy stabilizes or starts to decrease

In [ ]:
# Visualize how k affects classification accuracy

### Step 4: Detailed Evaluation of the Best Model

Now we evaluate the optimal kNN model using multiple metrics:

**Confusion Matrix** shows:
- **True Positives (TP)**: Correctly identified as RR Lyrae
- **True Negatives (TN)**: Correctly identified as Normal Star  
- **False Positives (FP)**: Incorrectly classified as RR Lyrae (Type I error)
- **False Negatives (FN)**: Incorrectly classified as Normal Star (Type II error)

**Key Metrics**:
- **Precision**: Of all predicted RR Lyrae stars, how many were actually RR Lyrae? (TP / (TP + FP))
- **Recall (Sensitivity)**: Of all actual RR Lyrae stars, how many did we correctly identify? (TP / (TP + FN))
- **F1-Score**: Harmonic mean of precision and recall - a balanced metric
- **Support**: Number of samples in each class

In [ ]:
# Train the best kNN model and evaluate in detail

# Make predictions

# Calculate confusion matrix

# Plot confusion matrix

# Classification Report

-----

# Decision Trees Classification

## Introduction to Decision Trees

A **Decision Tree** is a tree-like model that makes predictions by recursively splitting the feature space based on feature values.

**How it works**:
1. The algorithm starts with all data at the root node
2. It selects the feature and threshold that best separates the classes (usually based on Gini impurity or information gain)
3. It recursively splits each node until reaching a stopping criterion
4. Leaf nodes represent class predictions

**Key Concept - Tree Depth**:
- **Shallow trees (small max_depth)**: Simple rules, fast training, may underfit
- **Deep trees (large max_depth)**: Complex rules, can capture intricate patterns, may overfit
- **Optimal depth**: Balances between bias and variance, generalizes well to new data

**Advantages over kNN**:
- Provides **interpretable rules** (you can see exactly how decisions are made)
- Works with both numerical and categorical features
- Doesn't require scaling of features
- Much faster predictions for large datasets

We'll use the same stellar dataset to classify stars as Normal or RR Lyrae using Decision Trees.

### Step 1: Train Decision Trees with Different Depths

We systematically train Decision Trees with maximum depths ranging from 1 to 20. The `max_depth` parameter controls the complexity:
- **max_depth=1**: A single split (stumpy tree, very simple)
- **max_depth=10-15**: Usually where optimal models live
- **max_depth=20+**: Very deep trees that may memorize training data

Like with kNN, we track both training and testing accuracy to detect overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

# Test different tree depths

# Find the best max_depth value

### Step 2: Visualize the Effect of Tree Depth on Model Performance

This plot compares Decision Tree performance across different depths:
- **Blue line with circles**: Training accuracy (usually stays high or reaches 100% with deep trees)
- **Orange line with squares**: Testing accuracy (the metric we care about most)
- **Red dashed line**: Marks the optimal tree depth

Notice the classic overfitting pattern:
- Training accuracy often reaches 100% for deep trees
- Testing accuracy peaks at an optimal depth, then may decline as the tree becomes too complex
- The gap between training and testing accuracy indicates overfitting severity

In [ ]:
# Visualize how max_depth affects classification accuracy

### Step 3: Detailed Evaluation of the Best Decision Tree Model

The same evaluation metrics as the kNN model apply here:

**Confusion Matrix** shows the classification outcomes:
- **True Positives (TP)**: Correctly identified as RR Lyrae
- **True Negatives (TN)**: Correctly identified as Normal Star  
- **False Positives (FP)**: Incorrectly classified as RR Lyrae
- **False Negatives (FN)**: Incorrectly classified as Normal Star

**Key Metrics**:
- **Precision**: Of all predicted RR Lyrae, how many were actually correct?
- **Recall**: Of all actual RR Lyrae stars, how many did we catch?
- **F1-Score**: Balanced average of precision and recall
- **Support**: Number of samples in each class in the test set

In [ ]:
# Train the best Decision Tree model and evaluate in detail

# Make predictions

# Calculate confusion matrix

# Plot confusion matrix

# Classification Report

### Step 4: Visualizing Decision Tree Structures

One of the main advantages of Decision Trees is **interpretability** - we can actually see the decision rules!

The following visualization shows the actual tree structure for three different depths:
- **max_depth=3**: A shallow tree with simple rules
- **max_depth=best_depth**: The optimal tree we selected
- **max_depth=10**: A deeper tree with more complex decision rules

Each node shows:
- **The condition** (feature and threshold) used to split the data
- **Gini impurity** (measure of data purity at that node; 0 means pure, 0.5 means mixed)
- **Sample count** and **value** (how many samples of each class)
- **Color intensity** (darker = more samples of one class, lighter = mixed)

In [1]:
# Visualize decision trees with different depths

### Step 5: k-Fold Cross Validation for Robust Hyperparameter Selection

In the previous approach, we split the data once into training and testing sets. However, the results can be **sensitive to how the data is split**. A more robust approach is **k-Fold Cross Validation**:

**How k-Fold Cross Validation Works**:
1. Divide the entire dataset into k equal-sized folds (e.g., 5 folds)
2. For each fold:
   - Use that fold as the **test set**
   - Use all other folds as the **training set**
   - Train the model and record the accuracy
3. Calculate the **mean accuracy** and **standard deviation** across all folds
4. The standard deviation tells us how stable the model is across different data splits

**Advantages**:
- **More reliable estimates**: Uses all data for both training and testing
- **Stability assessment**: Standard deviation shows if performance varies across folds
- **Better use of limited data**: No data is wasted (unlike a single train-test split)
- **Reduces variance**: Multiple splits give a more robust estimate of true performance

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# Use 5-fold cross validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Test different tree depths using cross validation

In [ ]:
# Train the best Decision Tree model and evaluate in detail

# Make predictions

# Calculate confusion matrix

# Plot confusion matrix

# Classification Report

------

# Random Forest Classification

## Introduction to Random Forests

A **Random Forest** is an ensemble learning method that combines multiple decision trees to create a more powerful and robust classifier.

**How it works**:
1. Creates multiple decision trees, each trained on a random subset of the data (with replacement - "bootstrap samples")
2. Each tree also uses a random subset of features at each split
3. For predictions, each tree "votes" and the majority class wins
4. This ensemble approach reduces overfitting compared to a single tree

**Key Advantages**:
- **Reduces overfitting**: Averaging predictions from multiple trees is more robust than a single tree
- **Handles high-dimensional data**: Random feature selection at each split
- **Provides feature importance**: Shows which features are most important for classification
- **Parallelizable**: Trees can be built independently in parallel
- **Out-of-bag (OOB) error**: Can estimate performance without a separate test set

**Key Hyperparameters**:
- **n_estimators**: Number of trees in the forest (more trees = better, but slower)
- **max_depth**: Maximum depth of each tree (controls complexity of individual trees)
- **min_samples_split**: Minimum samples required to split a node (controls tree growth)

We'll use the same stellar dataset to classify stars as Normal or RR Lyrae using Random Forests.

### Step 1: Train Random Forests with Different Numbers of Trees

We'll test Random Forests with different numbers of trees (n_estimators) ranging from 1 to 200.

**Why n_estimators matters**:
- **n_estimators=1**: Just a single random tree, similar to a decision tree
- **n_estimators=10-50**: Good balance between performance and computational cost
- **n_estimators=100+**: More robust predictions, but diminishing returns
- **Very large n_estimators**: More stable predictions but slower training and prediction

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Test different numbers of trees
n_estimators_range = []
rf_train_accuracies = []
rf_test_accuracies = []

print("Training Random Forests with different numbers of trees...\n")

### Step 2: Visualize the Effect of Number of Trees on Performance

This plot shows how the number of trees affects model performance:
- **Blue line with circles**: Training accuracy (usually stays high or reaches 100%)
- **Orange line with squares**: Testing accuracy (what we care about most)

Key observations:
- Performance typically improves quickly as we add more trees
- After a certain point, adding more trees shows diminishing returns
- Unlike single decision trees, Random Forests typically don't overfit as much

In [ ]:
# Visualize how n_estimators affects Random Forest performance

### Step 3: Detailed Evaluation of the Best Random Forest Model

The confusion matrix and classification report provide detailed insights into the model's performance.

In [2]:
# Train the best Random Forest model and evaluate in detail

# Train the best Random Forest model

# Calculate confusion matrix

# Classification Report

### Step 4: Feature Importance Analysis

One of the key advantages of Random Forests is that they provide **feature importance** - a measure of how much each feature contributes to the classification.

The importance is based on how much each feature decreases the impurity (Gini or entropy) across all trees in the forest. Features that are more important for separating the classes will have higher importance scores.

In [ ]:
# Extract feature importances

# Print feature importances

# Visualize feature importance

### Step 5: Cross Validation Evaluation of Random Forest

Let's use k-fold cross validation to robustly evaluate the Random Forest model with different numbers of trees.

In [ ]:
# 5-fold cross validation for Random Forest

# Find the best n_estimators using cross validation

In [ ]:
# Visualize cross validation results

## Step 6. Grid Search for Hyperparameter Tuning

### What is Grid Search?

**Grid Search** is a systematic approach to hyperparameter tuning that exhaustively searches through all combinations of specified hyperparameter values. It's called "grid" because you define a grid of values for each parameter, and the search tests all possible combinations.

### How Grid Search Works

1. **Define Parameter Grid**: Specify ranges or lists of values for each hyperparameter
2. **Create All Combinations**: Generate all possible combinations of parameter values
3. **Cross Validation**: For each combination, evaluate the model using k-fold cross validation
4. **Select Best**: Choose the combination with the best cross-validation score
5. **Final Training**: Train a final model using the best parameters on the full training set

### Key Hyperparameters for Random Forest

- **n_estimators**: Number of decision trees in the forest (more ≠ always better)
- **max_depth**: Maximum depth of each tree (controls complexity)
- **min_samples_split**: Minimum samples required to split a node (prevents small splits)
- **min_samples_leaf**: Minimum samples required at a leaf node (controls final predictions)
- **max_features**: Number of features considered at each split ('sqrt' or 'log2' are common)
- **bootstrap**: Whether to use bootstrap samples for training trees

### Trade-offs in Hyperparameter Selection

| Parameter | Increasing Value | Effect |
|-----------|------------------|--------|
| n_estimators | More trees | Better performance but slower |
| max_depth | Deeper trees | More complex model, risk of overfitting |
| min_samples_split | Higher values | Simpler trees, less overfitting |
| min_samples_leaf | Higher values | Simpler leaf nodes, less overfitting |
| max_features | More features | More complex splits, potentially better patterns |
| bootstrap | True | More diversity, less overfitting |

### Why Use Grid Search?

1. **Finds Parameter Interactions**: Some parameters work better together
2. **Systematic Exploration**: Ensures you don't miss good combinations
3. **Robust Evaluation**: Cross validation gives stable performance estimates
4. **Reproducible**: Results can be replicated and documented
5. **Scalable**: Can be parallelized for faster computation

### When to Use Grid Search vs Other Methods

- **Grid Search**: Good for small to medium parameter spaces, when you have computational resources
- **Random Search**: Better for large parameter spaces, samples random combinations
- **Bayesian Optimization**: For complex spaces where past results guide future searches
- **Manual Tuning**: Only when you have strong prior knowledge about good parameters

In [ ]:
# Grid Search for Random Forest Hyperparameter Optimization
# This performs an exhaustive search over specified parameter values for a Random Forest classifier

# Define the parameter grid to search over
param_grid = {
    'n_estimators': [100, 250, 500],           # Number of trees
    'max_depth': [3, 5, 10, 15, None],            # Maximum tree depth (None = unlimited)
    'min_samples_split': [5, 10],               # Minimum samples to split a node
    'max_features': ['sqrt', 'log2'],              # Number of features to consider at each split
    'bootstrap': [True]                     # Whether to use bootstrap samples
}


# Create the base Random Forest classifier

# Perform Grid Search with Cross Validation

# Fit the grid search

# Extract the best model

# Test the best model on the test set

In [ ]:
# Analyze Grid Search Results

In [ ]:
# Detailed Evaluation of Best Model from Grid Search
# Create confusion matrix and classification report

In [ ]:
# Comparison: Basic Tuning vs Grid Search
# Compare the model trained with basic n_estimators tuning vs comprehensive grid search

------

# XGBoost Classification

## Introduction to XGBoost (Extreme Gradient Boosting)

**XGBoost** is a highly optimized gradient boosting framework that has become one of the most popular machine learning algorithms for classification and regression tasks.

**How it works**:
1. Builds an ensemble of decision trees sequentially, with each new tree correcting errors made by previous trees
2. Uses gradient descent to minimize a loss function
3. Applies regularization to prevent overfitting (L1 and L2 penalties)
4. Features parallel and distributed computing for speed
5. Uses a technique called "shrinkage" (learning rate) to gradually improve predictions

**Key Advantages**:
- **Superior Performance**: Often achieves state-of-the-art results on classification and regression tasks
- **Handles Imbalanced Data**: Built-in support for `scale_pos_weight` to handle class imbalance
- **Feature Importance**: Provides detailed feature importance rankings
- **Regularization**: L1/L2 regularization reduces overfitting
- **Speed**: Highly optimized and can be parallelized
- **Stability**: More stable predictions than single decision trees

**Key Hyperparameters**:
- **n_estimators**: Number of boosting rounds (trees to build)
- **max_depth**: Maximum depth of each tree (typically 3-10 for XGBoost)
- **learning_rate**: Shrinkage parameter controlling how much each tree contributes (0.01-0.3)
- **min_child_weight**: Minimum sum of weights needed in a child node
- **subsample**: Fraction of samples used for building each tree
- **colsample_bytree**: Fraction of features used for building each tree
- **scale_pos_weight**: Weight for imbalanced classes (important for our RR Lyrae problem!)

We'll optimize these hyperparameters using k-fold cross validation on the stellar classification task.

### Step 1: Hyperparameter Tuning Using k-Fold Cross Validation

XGBoost has many hyperparameters. We'll optimize the most important ones using k-fold cross validation:
- **max_depth**: Controls tree complexity (3-10)
- **learning_rate**: Controls how fast the model learns (0.01-0.3)
- **n_estimators**: Number of boosting rounds (50-500)

We'll test combinations of these and track balanced accuracy to find the best configuration.

In [ ]:
# Import XGBoost
import xgboost as xgb

In [ ]:
# Visualize grid search results

### Step 2: Detailed Evaluation of the Best XGBoost Model

Now we evaluate the optimized XGBoost model on the test set using our comprehensive metrics.

In [ ]:
# Make predictions with the best XGBoost model

# Calculate confusion matrix

# Plot confusion matrix

# Classification Report

# Calculate metrics for comparison

### Step 3: Feature Importance Analysis for XGBoost

XGBoost provides detailed feature importance information showing how many times each feature was used for splitting across all trees.

In [ ]:
# Extract feature importances from XGBoost

# Print feature importances

# Visualize feature importance

-----

# Comprehensive Algorithm Comparison - All Methods

Now let's compare all four classification algorithms: kNN, Decision Tree, Random Forest, and XGBoost.

In [ ]:
# Calculate metrics for all algorithms
# kNN

# Decision Tree

# Random Forest

# Create comprehensive comparison dataframe